# CoroVision ARCADE stenosis: individual training
Downloads the official release, verifies its published MD5, validates both tasks, performs EDA, then trains only **stenosis** using original polygons and source classes. Select a free T4 GPU runtime. Download checkpoints before the runtime expires. Official test data is audited but never used for fitting or selection. No severity labels are invented. Dataset: https://zenodo.org/records/10390295

In [ ]:
import base64, io, zipfile, pathlib, subprocess, sys, hashlib, json
PROJECT = pathlib.Path('/content/corovision_individual')
PROJECT.mkdir(exist_ok=True)
PAYLOAD = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    archive.extractall(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'Pillow>=10', 'numpy', 'shapely>=2', 'opencv-python-headless'], check=True)
import torch, torchvision
print('Torch', torch.__version__, 'CUDA', torch.cuda.is_available())


In [ ]:
archive_path = PROJECT / 'arcade.zip'
subprocess.run(['curl', '-fL', '--retry', '6', '--retry-all-errors', '-C', '-', '-o', str(archive_path), 'https://zenodo.org/records/10390295/files/arcade.zip?download=1'], check=True)
digest = hashlib.md5()
with archive_path.open('rb') as f:
    for block in iter(lambda: f.read(1024*1024), b''): digest.update(block)
assert digest.hexdigest() == 'c5b1973ade06f7dff210f878161e1a76', 'Archive checksum mismatch'
with zipfile.ZipFile(archive_path) as z:
    for member in z.infolist():
        target = (PROJECT / member.filename).resolve()
        assert target.is_relative_to(PROJECT.resolve()), 'Unsafe archive path'
    assert z.testzip() is None, 'ZIP CRC validation failed'
    z.extractall(PROJECT)
assert (PROJECT / 'arcade/syntax/train/images').is_dir(), 'Unexpected dataset layout'


In [ ]:
subprocess.run([sys.executable, '-m', 'ml.preprocessing.run_preprocessing', '--data-root', 'arcade', '--output-root', 'data/processed/arcade'], cwd=PROJECT, check=True)
report = json.loads((PROJECT/'data/processed/arcade/reports/inventory.json').read_text())
assert report['status'] != 'fail', 'Preprocessing failed: training blocked'
subprocess.run([sys.executable, 'scripts/run_dataset_eda.py', '--data-root', 'arcade', '--output-root', 'outputs/eda'], cwd=PROJECT, check=True)
print(json.dumps(report['issue_counts'], indent=2))


## Training gate
Preprocessing and EDA must succeed first. Training uses train and validation only. Decoded-image duplicates are explicitly excluded from the training loader while validation remains fixed. Checkpoints, per-class validation scores and logs are stored per task/architecture. The test set is reserved for final evaluation after model selection.

In [ ]:
assert torch.cuda.is_available(), 'Training blocked: select a free T4 GPU runtime'
print('GPU:', torch.cuda.get_device_name(0))
TASK = 'stenosis'
subprocess.run([sys.executable, 'scripts/train_arcade_instances.py', '--task', TASK, '--architectures', 'maskrcnn_resnet50_fpn', '--epochs', '25', '--batch-size', '2', '--workers', '2', '--seed', '42', '--patience', '6', '--output-dir', f'models/individual/{TASK}'], cwd=PROJECT, check=True)


In [ ]:
import shutil
from google.colab import files
result_path = shutil.make_archive(f'/content/corovision_{TASK}_results', 'zip', PROJECT, f'models/individual/{TASK}')
files.download(result_path)
